# PREPROCESSING AND EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# Load necessary libraries

import os 
import sys

import pandas as pd
import numpy as np
import scipy.sparse as sp
import json

: 

In [ ]:
os.chdir("../")
sys.path.append(os.path.abspath(".."))

In [ ]:
movies_path = os.path.join("data", "raw", "ml-1m", "movies.dat")
rating_path = os.path.join("data", "raw", "ml-1m", "ratings.dat")
users_path = os.path.join("data", "raw", "ml-1m", "users.dat")

ratings_df = pd.read_csv(rating_path, sep="::", engine="python", header=None, names=["user_id", "movie_id", "rating", "timestamp"])
movies_df = pd.read_csv(movies_path, sep="::", engine="python", header=None, names=["movieid", "title", "genres"], encoding="latin-1")
user_df = pd.read_csv(users_path, sep="::", engine= "python", header= None, names=["user_id","gender","age","occupation", "zipcode"])

In [ ]:
ratings_df.head()

In [ ]:
movies_df.head()

In [ ]:
user_df.head()

## Rating Transformation

In [ ]:
ratings_df.head()

In [ ]:
ratings_count = ratings_df.groupby("rating")["movie_id"].count()
ratings_count

In [ ]:
movie_count = ratings_df.groupby("movie_id")["rating"].count()
movie_count

In [ ]:
print(f"Ratings shape before filtering: {ratings_df.shape}")
print(f"Unique movies before filtering: {ratings_df['movie_id'].nunique()}")

In [ ]:
valid_movies = movie_count[movie_count >= 10].index
ratings_df = ratings_df[ratings_df["movie_id"].isin(valid_movies)]

print(f"Ratings shape after filtering: {ratings_df.shape}")
print(f"Unique movies after filtering: {ratings_df['movie_id'].nunique()}")

## SPILT DATSET INTO TRAIN AND TEST

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42,
    stratify=ratings_df["user_id"]
)

print("Train size:", train_data.shape)
print("Test size:", test_data.shape)

In [ ]:
user_ids = train_data["user_id"].unique()
movie_ids = train_data["movie_id"].unique()

user_map = {uid: idx for idx, uid in enumerate(user_ids)}
item_map = {mid: idx for idx, mid in enumerate(movie_ids)}

print("Total users:", len(user_map))
print("Total movies:", len(item_map))

In [ ]:
type(user_map), type(item_map)

## Build Matrix with User and Item Indexes

In [ ]:
from scipy.sparse import csr_matrix

row = train_data["user_id"].map(user_map)
col = train_data["movie_id"].map(item_map)
values = train_data["rating"].values

train_matrix = csr_matrix(
    (values, (row, col)),
    shape=(len(user_map), len(item_map))
)

print("Matrix shape:", train_matrix.shape)
print("Total ratings stored:", train_matrix.nnz)
print("Sparsity:", round(1 - train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1]), 4))

In [ ]:
total_matrix_size = len(user_map) * len(item_map)
print(f"Total possible ratings: {total_matrix_size}")

total_ratings_stored = train_matrix.nnz
print(f"Total ratings stored: {total_ratings_stored}")

empty_cells = total_matrix_size - total_ratings_stored
print(f"Empty cells: {empty_cells}")

filled_percentage = total_ratings_stored / total_matrix_size * 100
print(f"Filled percentage: {filled_percentage:.4f}%")

sparsity = (1 - train_matrix.nnz / total_matrix_size) * 100
print(f"Sparsity: {sparsity:.4f}%")

* Matrix sparsity is 95.94% — only 4% of cells have ratings.
* This is the core challenge of recommendation systems.

In [ ]:
from scipy.sparse import save_npz

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/processed/mappings", exist_ok=True)

# save the sparse matrix
save_npz("../data/processed/train_matrix.npz", train_matrix)

# save test data
test_data.to_csv("../data/processed/test_data.csv", index=False)

# save mappings
pd.DataFrame(list(user_map.items()), columns=["user_id", "user_idx"]).to_csv("../data/processed/mappings/user_map.csv", index=False)
pd.DataFrame(list(item_map.items()), columns=["movie_id", "item_idx"]).to_csv("../data/processed/mappings/item_map.csv", index=False)